# SVM Experiment — PlacePredictor

**Student:** Muntasir Bin Kashem | **ID:** 0112331000 | **Branch:** svm_0112331000

Interactive notebook for the Support Vector Machine placement prediction module.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import config
from models.svm_pipeline import build_svm_pipeline, get_baseline_svm, get_tuning_param_grid
from models.evaluate_svm import run_full_evaluation
from models.shap_analysis import run_shap_analysis, generate_shap_interpretation
from models.train_svm import get_cv_scorer
from sklearn.model_selection import GridSearchCV, train_test_split

print(f"Project root: {PROJECT_ROOT}")

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv(config.DATA_PATH)
df = df.drop(columns=[c for c in config.DROP_COLUMNS if c in df.columns])

print(f"Shape: {df.shape}")
print(f"\nTarget distribution:\n{df[config.TARGET_COLUMN].value_counts()}")
df.head()

## 2. Train/Test Split

In [ ]:
X = df[config.FEATURE_COLUMNS]
y = df[config.TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 3. Baseline SVM

In [ ]:
baseline = build_svm_pipeline(get_baseline_svm())
baseline.fit(X_train, y_train)
baseline_results = run_full_evaluation(baseline, X_test, y_test, "Baseline SVM", "baseline_svm")
baseline_results["metrics"]

## 4. Hyperparameter Tuning (GridSearchCV)

In [ ]:
pipeline = build_svm_pipeline(get_baseline_svm())
grid = GridSearchCV(
    pipeline,
    get_tuning_param_grid(),
    cv=config.CV_FOLDS,
    scoring=get_cv_scorer(),
    n_jobs=1,
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.4f}")

## 5. Tuned SVM Evaluation

In [ ]:
tuned = grid.best_estimator_
tuned_results = run_full_evaluation(tuned, X_test, y_test, "Tuned SVM", "tuned_svm")
tuned_results["metrics"]

## 6. SHAP Explainability

In [ ]:
importance_df = run_shap_analysis(tuned, X_train, X_test)
print(generate_shap_interpretation(importance_df))
importance_df.head(10)

## 7. View Generated Plots

All plots are saved under `results/plots/`.

In [ ]:
from IPython.display import Image, display

for plot in sorted(config.PLOTS_DIR.glob("*.png")):
    print(plot.name)
    display(Image(filename=str(plot)))